In [10]:
import pandas as pd
import numpy as np
import random
from sklearn.preprocessing import StandardScaler, LabelEncoder
# Import pour le téléchargement automatique dans Google Colab
from google.colab import files

def analyze_prepare_and_download_fl_csv():
    # 1. FIXER LES GRAINES ALÉATOIRES
    def set_seed(seed=42):
        np.random.seed(seed)
        random.seed(seed)

    set_seed(42)
    print("[INFO] Graines aléatoires fixées.")

    # 2. CHARGEMENT DES DONNÉES
    train_df = pd.read_csv('UNSW_NB15_training-set.csv')
    test_df = pd.read_csv('UNSW_NB15_testing-set.csv')

    # Nettoyage préventif des espaces textuels dans les catégories d'attaques
    train_df['attack_cat'] = train_df['attack_cat'].str.strip()
    test_df['attack_cat'] = test_df['attack_cat'].str.strip()

    total_train = len(train_df)
    total_test = len(test_df)

    # 3. ENCODAGE DE ATTACK_CAT (Cible principale multi-classe)
    le_attack = LabelEncoder()
    combined_attacks = pd.concat([train_df['attack_cat'], test_df['attack_cat']]).astype(str)
    le_attack.fit(combined_attacks)

    train_df['target'] = le_attack.transform(train_df['attack_cat'].astype(str))
    test_df['target'] = le_attack.transform(test_df['attack_cat'].astype(str))

    # Suppression des colonnes inutiles ou redondantes pour l'apprentissage
    cols_to_drop = ['id', 'label', 'Label']
    X_train_df = train_df.drop(columns=[c for c in cols_to_drop if c in train_df.columns])
    X_test_df = test_df.drop(columns=[c for c in cols_to_drop if c in test_df.columns])

    # 4. ENCODAGE DES VARIABLES CATÉGORIQUES (Features : proto, service, state)
    cat_cols = ['proto', 'service', 'state']
    for col in cat_cols:
        le = LabelEncoder()
        combined_data = pd.concat([X_train_df[col], X_test_df[col]]).astype(str)
        le.fit(combined_data)
        X_train_df[col] = le.transform(X_train_df[col].astype(str))
        X_test_df[col] = le.transform(X_test_df[col].astype(str))

    # 5. NORMALISATION DES FEATURES NUMÉRIQUES
    # On isole les colonnes de caractéristiques à normaliser
    features_to_scale = X_train_df.drop(columns=['attack_cat', 'target'], errors='ignore')
    num_cols = features_to_scale.select_dtypes(include=['int64', 'float64', 'int32']).columns

    scaler = StandardScaler()
    X_train_df[num_cols] = scaler.fit_transform(X_train_df[num_cols])
    X_test_df[num_cols] = scaler.transform(X_test_df[num_cols])

    # =========================================================================
    # AFFICHAGE ANALYSE 1 : POURCENTAGES DANS LE DATASET GLOBAL (TRAIN & TEST)
    # =========================================================================
    print("\n" + "="*75)
    print(" ANALYSE DES ATTAQUES : DATASET TRAIN GLOBAL vs DATASET TEST GLOBAL")
    print("="*75)
    print(f"{'Type d\'Attaque':<20} | {'Train (Qté)':<12} | {'Train (%)':<10} | {'Test (Qté)':<12} | {'Test (%)':<10}")
    print("-" * 75)

    all_unique_classes = sorted(combined_attacks.unique())
    for cat in all_unique_classes:
        count_train = np.sum(train_df['attack_cat'] == cat)
        pct_train = (count_train / total_train) * 100

        count_test = np.sum(test_df['attack_cat'] == cat)
        pct_test = (count_test / total_test) * 100

        print(f"{cat:<20} | {count_train:<12} | {pct_train:<10.2f}% | {count_test:<12} | {pct_test:<10.2f}%")
    print("="*75)

    # =========================================================================
    # RÉPARTITION FL : PRÉPARATION DU TRAFFIC 'NORMAL' DIVISÉ EN 3
    # =========================================================================
    normal_indices = train_df[train_df['attack_cat'].str.lower() == 'normal'].index.tolist()
    random.seed(42)
    random.shuffle(normal_indices)

    split_size = len(normal_indices) // 3
    normal_split_0 = normal_indices[:split_size]
    normal_split_1 = normal_indices[split_size:2*split_size]
    normal_split_2 = normal_indices[2*split_size:]

    # Cartographie demandée des attaques spécifiques
    client_attack_mapping = {
        0: ['Fuzzers', 'DoS', 'Reconnaissance'],
        1: ['Generic', 'Backdoor', 'Worms'],
        2: ['Exploits', 'Analysis', 'Shellcode']
    }

    client_normal_mapping = {
        0: normal_split_0,
        1: normal_split_1,
        2: normal_split_2
    }

    print("\n" + "="*75)
    print(" SIMULATION FL : DISTRIBUTION DES ATTAQUES SUR 3 CLIENTS (NON-IID + NORMAL/3)")
    print("="*75)

    csv_files_to_download = []

    for client_id in range(3):
        specific_attacks = client_attack_mapping[client_id]

        # Sélection des données des attaques spécifiques et récupération des données Normales
        client_attack_df = train_df[train_df['attack_cat'].isin(specific_attacks)]
        attack_indices = client_attack_df.index.tolist()
        assigned_normal_indices = client_normal_mapping[client_id]

        # Fusion des indices pour ce client
        client_all_indices = attack_indices + assigned_normal_indices

        # Création du DataFrame final pré-traité pour le client (Features normalisées + Target numérique)
        # On retire 'attack_cat' (texte) pour n'avoir que du numérique propre pour vos modèles
        client_final_csv_df = X_train_df.loc[client_all_indices].drop(columns=['attack_cat'], errors='ignore')

        client_samples = len(client_final_csv_df)
        pct_of_global_train = (client_samples / total_train) * 100

        print(f"\n▶ CLIENT {client_id}")
        print(f"  Volume total de données locales : {client_samples} lignes ({pct_of_global_train:.2f}% du train global)")
        print(f"  Attaques spécifiques assignées  : {specific_attacks} + [Normal]")
        print(f"  -----------------------------------------------------------------")
        print(f"  {'Attaque Contenue':<20} | {'Qté Locale':<12} | {'% Interne Client':<18} | {'% du Train Global':<18}")
        print(f"  -----------------------------------------------------------------")

        # Affichage statistiques : Trafic Normal
        count_normal = len(assigned_normal_indices)
        pct_interne_normal = (count_normal / client_samples) * 100 if client_samples > 0 else 0
        pct_global_normal = (count_normal / total_train) * 100
        print(f"  {'Normal':<20} | {count_normal:<12} | {pct_interne_normal:<18.2f}% | {pct_global_normal:<18.2f}%")

        # Affichage statistiques : Chaque attaque spécifique
        for attack in specific_attacks:
            count_local = np.sum(train_df.loc[client_all_indices, 'attack_cat'] == attack)
            pct_interne = (count_local / client_samples) * 100 if client_samples > 0 else 0
            pct_global = (count_local / total_train) * 100
            print(f"  {attack:<20} | {count_local:<12} | {pct_interne:<18.2f}% | {pct_global:<18.2f}%")

        # --- SAUVEGARDE EN CSV LOCAL ---
        filename = f'client_{client_id}_train.csv'
        client_final_csv_df.to_csv(filename, index=False)
        csv_files_to_download.append(filename)
        print(f"  -> [GÉNÉRATION CSV] Réussie : '{filename}' (Shape: {client_final_csv_df.shape})")
        print("-" * 75)

    # =========================================================================
    # PRÉPARATION ET GÉNÉRATION DU TEST SET GLOBAL EN CSV
    # =========================================================================
    test_final_csv_df = X_test_df.drop(columns=['attack_cat'], errors='ignore')
    test_filename = 'global_test.csv'
    test_final_csv_df.to_csv(test_filename, index=False)
    csv_files_to_download.append(test_filename)

    print("\n" + "="*75)
    print(f"[INFO] -> [GÉNÉRATION CSV] Réussie : '{test_filename}' (Shape: {test_final_csv_df.shape})")
    print("="*75)

    # =========================================================================
    # LANCE LE TÉLÉCHARGEMENT AUTOMATIQUE VIA LE NAVIGATEUR (SPÉCIFIQUE COLAB)
    # =========================================================================
    print("\n[INFO] Lancement des téléchargements vers votre machine locale...")
    for file in csv_files_to_download:
        print(f"Téléchargement en cours de : {file}")
        files.download(file)

    print("\n[INFO] Traitement terminé. Vérifiez la barre de téléchargement de votre navigateur.")
    print("="*75 + "\n")
analyze_prepare_and_download_fl_csv()

# Lancer la fonction complète dans votre cellule Colab
# analyze_prepare_and_download_fl_csv()

[INFO] Graines aléatoires fixées.

 ANALYSE DES ATTAQUES : DATASET TRAIN GLOBAL vs DATASET TEST GLOBAL
Type d'Attaque       | Train (Qté)  | Train (%)  | Test (Qté)   | Test (%)  
---------------------------------------------------------------------------
Analysis             | 2000         | 1.14      % | 677          | 0.82      %
Backdoor             | 1746         | 1.00      % | 583          | 0.71      %
DoS                  | 12264        | 6.99      % | 4089         | 4.97      %
Exploits             | 33393        | 19.04     % | 11132        | 13.52     %
Fuzzers              | 18184        | 10.37     % | 6062         | 7.36      %
Generic              | 40000        | 22.81     % | 18871        | 22.92     %
Normal               | 56000        | 31.94     % | 37000        | 44.94     %
Reconnaissance       | 10491        | 5.98      % | 3496         | 4.25      %
Shellcode            | 1133         | 0.65      % | 378          | 0.46      %
Worms                | 130       

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Téléchargement en cours de : client_1_train.csv


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Téléchargement en cours de : global_test.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


[INFO] Traitement terminé. Vérifiez la barre de téléchargement de votre navigateur.

